In [1]:
# ==========================================
# PRAKTIKUM 1: Membandingkan Feature Selection
# ==========================================

import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression, Lasso

# 1. Load Dataset (Menggunakan Breast Cancer Dataset sebagai contoh)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Bagi data menjadi Train dan Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standarisasi fitur (Sangat penting untuk Lasso dan RFE)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

print(f"Jumlah fitur awal: {X.shape[1]}\n")
print("-" * 50)

# ==========================================
# METODE 1: FILTER (SelectKBest)
# ==========================================
# Memilih 5 fitur terbaik berdasarkan skor ANOVA F-value
selector_filter = SelectKBest(score_func=f_classif, k=5)
selector_filter.fit(X_train_scaled, y_train)

# Mendapatkan nama fitur yang terpilih
fitur_filter = X.columns[selector_filter.get_support()]
print("1. FILTER METHOD (SelectKBest)")
print("Fitur terpilih:", list(fitur_filter))
print("-" * 50)

# ==========================================
# METODE 2: WRAPPER (RFE)
# ==========================================
# Menggunakan Logistic Regression sebagai estimator dasar
estimator = LogisticRegression(max_iter=1000)
selector_rfe = RFE(estimator=estimator, n_features_to_select=5, step=1)
selector_rfe.fit(X_train_scaled, y_train)

# Mendapatkan nama fitur yang terpilih
fitur_rfe = X.columns[selector_rfe.support_]
print("2. WRAPPER METHOD (RFE)")
print("Fitur terpilih:", list(fitur_rfe))
print("-" * 50)

# ==========================================
# METODE 3: EMBEDDED (Lasso)
# ==========================================
# Lasso (L1 Regularization) akan membuat koefisien fitur yang tidak penting menjadi 0
lasso = Lasso(alpha=0.05) # Alpha bisa disesuaikan
lasso.fit(X_train_scaled, y_train)

# Mendapatkan fitur yang koefisiennya tidak nol
fitur_lasso = X.columns[lasso.coef_ != 0]
print("3. EMBEDDED METHOD (Lasso)")
print("Fitur terpilih:", list(fitur_lasso))
print("-" * 50)

# ==========================================
# KESIMPULAN
# ==========================================
print("KESIMPULAN PERBANDINGAN:")
print(f"- Filter (SelectKBest) memilih {len(fitur_filter)} fitur.")
print(f"- Wrapper (RFE) memilih {len(fitur_rfe)} fitur.")
print(f"- Embedded (Lasso) memilih {len(fitur_lasso)} fitur.")

Jumlah fitur awal: 30

--------------------------------------------------
1. FILTER METHOD (SelectKBest)
Fitur terpilih: ['mean perimeter', 'mean concave points', 'worst radius', 'worst perimeter', 'worst concave points']
--------------------------------------------------
2. WRAPPER METHOD (RFE)
Fitur terpilih: ['radius error', 'worst radius', 'worst texture', 'worst area', 'worst concave points']
--------------------------------------------------
3. EMBEDDED METHOD (Lasso)
Fitur terpilih: ['mean concave points', 'worst radius', 'worst texture', 'worst concave points', 'worst symmetry']
--------------------------------------------------
KESIMPULAN PERBANDINGAN:
- Filter (SelectKBest) memilih 5 fitur.
- Wrapper (RFE) memilih 5 fitur.
- Embedded (Lasso) memilih 5 fitur.


In [2]:
# ==========================================
# PRAKTIKUM 2: Membangun Pipeline Lengkap
# ==========================================

import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Load Dataset Titanic
titanic = sns.load_dataset('titanic')

# Pilih fitur yang akan digunakan (sesuai instruksi: age, fare, sex, embarked)
features = ['age', 'fare', 'sex', 'embarked']
X = titanic[features]
y = titanic['survived']

# Bagi data menjadi Train dan Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 2. Rancang ColumnTransformer
# ==========================================
# Definisikan kolom
numeric_features = ['age', 'fare']
categorical_features = ['sex', 'embarked']

# Pipeline untuk data Numerik: Isi missing value (mean) -> Scaling
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Pipeline untuk data Kategorikal: Isi missing value (mode) -> OneHotEncoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Gabungkan dengan ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# ==========================================
# 3. Bungkus dengan Model Klasifikasi dalam Pipeline
# ==========================================
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# ==========================================
# 4. Latih Model & 5. Evaluasi Akurasi
# ==========================================
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

akurasi = accuracy_score(y_test, y_pred)

print("=== HASIL EVALUASI PIPELINE TITANIC ===")
print(f"Akurasi Model: {akurasi:.4f} ({akurasi*100:.2f}%)\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

=== HASIL EVALUASI PIPELINE TITANIC ===
Akurasi Model: 0.7765 (77.65%)

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.83      0.81       105
           1       0.74      0.70      0.72        74

    accuracy                           0.78       179
   macro avg       0.77      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179

